# Physical-Quantity Probe Table

This notebook isolates the physical-quantity probing analysis from `analyze_learned_representations.ipynb`.
It loads exported train/eval embeddings, trains probes on train embeddings only, evaluates on eval embeddings only, and builds a compact R2 table comparing methods.


In [ ]:
from pathlib import Path
import json
import os
import sys
import warnings

REPO_ROOT = Path.cwd().resolve()
for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
    if (candidate / "experiments").exists() and (candidate / ".git").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate repository root from current working directory.")

os.environ.setdefault("MPLCONFIGDIR", str(REPO_ROOT / ".mplconfig"))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)

EXPORT_ROOT = REPO_ROOT / "experiments" / "embedding_analysis" / "outputs"

ENVS = [
    {"name": "TwoRoom", "slug": "tworoom"},
    {"name": "Reacher", "slug": "reacher"},
    {"name": "Push-T", "slug": "pusht"},
    {"name": "OGBench-Cube", "slug": "ogbcube"},
]

METHODS = [
    {"name": "inverse", "label": "ours"},
    {"name": "sigreg", "label": "SIGReg"},
    {"name": "forward_only", "label": "forward-only"},
]

PROBE_TYPES = ["linear", "mlp"]
PROBE_LABELS = {"linear": "Linear probe", "mlp": "MLP probe"}

RUN_MLP_PROBE = True
MIN_TRAIN_ROWS = 20

# Controls only what appears in the final table. All entries in TARGET_SPECS are still probed.
DISPLAY_QUANTITY_GROUPS = {
    "tworoom": [
        "agent position",
    ],
    "reacher": [
        "joint angles / arm configuration",
    ],
    "pusht": [
        "agent position",
        "block position",
        "block orientation",
    ],
    "ogbcube": [
        "cube position",
        "cube orientation yaw",
        "end-effector position",
        "end-effector yaw",
        "gripper opening",
        "robot joint positions",
    ],
}

def pretty_path(path):
    path = Path(path)
    try:
        return path.relative_to(REPO_ROOT)
    except ValueError:
        return path


print(f"Repo root:   {REPO_ROOT}")
print(f"Export root: {EXPORT_ROOT}  (exists: {EXPORT_ROOT.exists()})")


In [ ]:
def torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def as_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def find_split_file(export_dir, split):
    export_dir = Path(export_dir)
    candidates = [
        export_dir / f"{split}_embeddings.pt",
        export_dir / f"{split}.pt",
    ]
    candidates.extend(sorted(export_dir.glob(f"*{split}*embedding*.pt")))
    candidates.extend(sorted(export_dir.glob(f"*{split}*.pt")))
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f"No {split} embedding file found in {export_dir}")


def load_split(export_dir, split):
    path = find_split_file(export_dir, split)
    payload = torch_load(path)
    if isinstance(payload, dict) and "embeddings" in payload:
        z = as_numpy(payload["embeddings"]).astype(np.float32)
        arrays = {k: as_numpy(v) for k, v in payload.get("arrays", {}).items()}
        metadata = payload.get("metadata", {})
        array_manifest = payload.get("array_manifest", {})
    else:
        z = as_numpy(payload).astype(np.float32)
        arrays = {}
        metadata = {}
        array_manifest = {}
    return {
        "path": path,
        "z": z,
        "arrays": arrays,
        "metadata": metadata,
        "array_manifest": array_manifest,
    }


def load_export(env_slug, method_name):
    export_dir = EXPORT_ROOT / env_slug / method_name
    train = load_split(export_dir, "train")
    eval_ = load_split(export_dir, "eval")
    array_keys = sorted(set(train["arrays"]) | set(eval_["arrays"]))
    metadata_keys = sorted(set(train["metadata"]) | set(eval_["metadata"]))
    print(f"\n[{env_slug}/{method_name}]")
    print(f"  train: {train['z'].shape}  {pretty_path(train['path'])}")
    print(f"  eval:  {eval_['z'].shape}  {pretty_path(eval_['path'])}")
    print(f"  array keys: {array_keys}")
    print(f"  metadata keys: {metadata_keys}")
    return {"train": train, "eval": eval_, "keys": array_keys, "dir": export_dir}


def load_all_exports():
    loaded = {}
    for env in ENVS:
        for method in METHODS:
            try:
                loaded[(env["slug"], method["name"])] = load_export(env["slug"], method["name"])
            except FileNotFoundError as exc:
                print(f"[skip export] {env['slug']}/{method['name']}: {exc}")
    return loaded


exports = load_all_exports()


In [ ]:

def array_2d(arrays, key):
    if key not in arrays:
        return None
    arr = np.asarray(arrays[key])
    if arr.ndim == 0:
        return arr.reshape(1, 1)
    if arr.ndim == 1:
        return arr.reshape(-1, 1)
    return arr.reshape(arr.shape[0], -1)


def first_available_matrix(arrays, candidates):
    for source, key, dims in candidates:
        arr = array_2d(arrays, key)
        if arr is None:
            continue
        dims = list(dims)
        if not dims or max(dims) >= arr.shape[1]:
            continue
        return source, arr[:, dims]
    return None, None


def first_available_component(arrays, candidates):
    for source, key, dim in candidates:
        arr = array_2d(arrays, key)
        if arr is None or dim >= arr.shape[1]:
            continue
        return source, arr[:, dim]
    return None, None


def angle_sincos(arrays, candidates):
    source, theta = first_available_component(arrays, candidates)
    if theta is None:
        return None, None
    theta = np.asarray(theta, dtype=np.float32).reshape(-1)
    return source, np.stack([np.sin(theta), np.cos(theta)], axis=1)


def angles_sincos(arrays, candidates):
    source, angles = first_available_matrix(arrays, candidates)
    if angles is None:
        return None, None
    angles = np.asarray(angles, dtype=np.float32)
    cols = []
    for dim in range(angles.shape[1]):
        cols.extend([np.sin(angles[:, dim]), np.cos(angles[:, dim])])
    return source, np.stack(cols, axis=1)


def default_components(spec, width):
    if "components" in spec:
        return spec["components"]
    if spec["kind"] == "angle":
        return ["sin(theta)", "cos(theta)"]
    if spec["kind"] == "angles_sincos":
        n_angles = width // 2
        names = []
        for i in range(n_angles):
            names.extend([f"sin(angle {i + 1})", f"cos(angle {i + 1})"])
        return names
    return [f"component {i + 1}" for i in range(width)]


def build_probe_target(arrays, spec):
    if spec["kind"] == "matrix":
        source, y = first_available_matrix(arrays, spec["candidates"])
    elif spec["kind"] == "angle":
        source, y = angle_sincos(arrays, spec["candidates"])
    elif spec["kind"] == "angles_sincos":
        source, y = angles_sincos(arrays, spec["candidates"])
    else:
        raise ValueError(f"Unknown target kind: {spec['kind']}")

    if y is None:
        return None
    y = np.asarray(y, dtype=np.float32).reshape(len(y), -1)
    return {
        "quantity": spec["quantity"],
        "source": source,
        "y": y,
        "components": default_components(spec, y.shape[1]),
    }


TARGET_SPECS = {
    "tworoom": [
        {
            "quantity": "agent position",
            "kind": "matrix",
            "candidates": [("pos_agent", "pos_agent", [0, 1]), ("proprio", "proprio", [0, 1]), ("observation", "observation", [0, 1])],
            "components": ["agent x", "agent y"],
        },
        {
            "quantity": "target position",
            "kind": "matrix",
            "candidates": [("pos_target", "pos_target", [0, 1]), ("observation", "observation", [2, 3])],
            "components": ["target x", "target y"],
        },
        {
            "quantity": "distance to target",
            "kind": "matrix",
            "candidates": [("distance_to_target", "distance_to_target", [0])],
            "components": ["distance"],
            "secondary": True,
        },
    ],
    "reacher": [
        {
            "quantity": "joint angles / arm configuration",
            "kind": "angles_sincos",
            "candidates": [("qpos", "qpos", [0, 1]), ("observation", "observation", [0, 1])],
            "components": ["sin joint 1", "cos joint 1", "sin joint 2", "cos joint 2"],
        },
        {
            "quantity": "end-effector position",
            "kind": "matrix",
            "candidates": [("finger_pos", "finger_pos", [0, 1])],
            "components": ["finger x", "finger y"],
        },
        {
            "quantity": "target position",
            "kind": "matrix",
            "candidates": [("target_pos", "target_pos", [0, 1])],
            "components": ["target x", "target y"],
        },
        {
            "quantity": "joint angular velocity",
            "kind": "matrix",
            "candidates": [("qvel", "qvel", [0, 1]), ("observation", "observation", [4, 5])],
            "components": ["joint velocity 1", "joint velocity 2"],
            "secondary": True,
        },
    ],
    "pusht": [
        {
            "quantity": "agent position",
            "kind": "matrix",
            "candidates": [("state", "state", [0, 1]), ("proprio", "proprio", [0, 1])],
            "components": ["agent x", "agent y"],
        },
        {
            "quantity": "block position",
            "kind": "matrix",
            "candidates": [("state", "state", [2, 3])],
            "components": ["block x", "block y"],
        },
        {
            "quantity": "block orientation",
            "kind": "angle",
            "candidates": [("state angle", "state", 4)],
            "components": ["sin block theta", "cos block theta"],
        },
        {
            "quantity": "agent velocity",
            "kind": "matrix",
            "candidates": [("state velocity", "state", [5, 6]), ("proprio", "proprio", [2, 3])],
            "components": ["agent vx", "agent vy"],
            "secondary": True,
        },
    ],
    "ogbcube": [
        {
            "quantity": "cube position",
            "kind": "matrix",
            "candidates": [("privileged_block_0_pos", "privileged_block_0_pos", [0, 1, 2])],
            "components": ["cube x", "cube y", "cube z"],
        },
        {
            "quantity": "cube orientation yaw",
            "kind": "angle",
            "candidates": [("privileged_block_0_yaw", "privileged_block_0_yaw", 0)],
            "components": ["sin cube yaw", "cos cube yaw"],
        },
        {
            "quantity": "end-effector position",
            "kind": "matrix",
            "candidates": [("proprio_effector_pos", "proprio_effector_pos", [0, 1, 2])],
            "components": ["ee x", "ee y", "ee z"],
        },
        {
            "quantity": "end-effector yaw",
            "kind": "angle",
            "candidates": [("proprio_effector_yaw", "proprio_effector_yaw", 0)],
            "components": ["sin ee yaw", "cos ee yaw"],
        },
        {
            "quantity": "gripper opening",
            "kind": "matrix",
            "candidates": [("proprio_gripper_opening", "proprio_gripper_opening", [0])],
            "components": ["opening"],
        },
        {
            "quantity": "gripper contact",
            "kind": "matrix",
            "candidates": [("proprio_gripper_contact", "proprio_gripper_contact", [0])],
            "components": ["contact"],
        },
        {
            "quantity": "robot joint positions",
            "kind": "matrix",
            "candidates": [("proprio_joint_pos", "proprio_joint_pos", [0, 1, 2, 3, 4, 5])],
            "components": [f"joint {i + 1}" for i in range(6)],
        },
        {
            "quantity": "robot joint velocities",
            "kind": "matrix",
            "candidates": [("proprio_joint_vel", "proprio_joint_vel", [0, 1, 2, 3, 4, 5])],
            "components": [f"joint velocity {i + 1}" for i in range(6)],
            "secondary": True,
        },
        {
            "quantity": "gripper velocity",
            "kind": "matrix",
            "candidates": [("proprio_gripper_vel", "proprio_gripper_vel", [0])],
            "components": ["gripper velocity"],
            "secondary": True,
        },
    ],
}


def target_inventory(env_slug, export):
    targets = []
    for spec in TARGET_SPECS[env_slug]:
        train_target = build_probe_target(export["train"]["arrays"], spec)
        eval_target = build_probe_target(export["eval"]["arrays"], spec)
        if train_target is None or eval_target is None:
            print(f"[skip target] {env_slug}: {spec['quantity']} unavailable")
            continue
        if train_target["y"].shape[1] != eval_target["y"].shape[1]:
            print(f"[skip target] {env_slug}: {spec['quantity']} train/eval dimensions differ")
            continue
        targets.append((spec, train_target, eval_target))
    return targets


In [ ]:
def finite_rows(*arrays):
    n = arrays[0].shape[0]
    mask = np.ones(n, dtype=bool)
    for arr in arrays:
        arr = np.asarray(arr)
        arr2 = arr.reshape(arr.shape[0], -1)
        mask &= np.isfinite(arr2).all(axis=1)
    return mask


def score_predictions(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(len(y_true), -1)
    y_pred = np.asarray(y_pred).reshape(len(y_pred), -1)
    component_r2 = r2_score(y_true, y_pred, multioutput="raw_values")
    grouped_r2 = r2_score(y_true, y_pred, multioutput="uniform_average")
    return float(grouped_r2), [float(x) for x in np.asarray(component_r2).reshape(-1)]


def fit_linear_probe(z_train, y_train, z_eval, y_eval):
    train_mask = finite_rows(z_train, y_train)
    eval_mask = finite_rows(z_eval, y_eval)
    if train_mask.sum() < MIN_TRAIN_ROWS or eval_mask.sum() == 0:
        raise ValueError(f"Not enough finite rows: train={train_mask.sum()}, eval={eval_mask.sum()}")

    model = make_pipeline(StandardScaler(), Ridge(alpha=1e-3))
    model.fit(z_train[train_mask], y_train[train_mask])
    pred = model.predict(z_eval[eval_mask])
    grouped_r2, component_r2 = score_predictions(y_eval[eval_mask], pred)
    return {
        "r2": grouped_r2,
        "component_r2": component_r2,
        "n_train": int(train_mask.sum()),
        "n_eval": int(eval_mask.sum()),
    }


def fit_mlp_probe(z_train, y_train, z_eval, y_eval):
    train_mask = finite_rows(z_train, y_train)
    eval_mask = finite_rows(z_eval, y_eval)
    if train_mask.sum() < MIN_TRAIN_ROWS or eval_mask.sum() == 0:
        raise ValueError(f"Not enough finite rows: train={train_mask.sum()}, eval={eval_mask.sum()}")

    x_scaler = StandardScaler()
    y_scaler = StandardScaler()
    x_train = x_scaler.fit_transform(z_train[train_mask])
    x_eval = x_scaler.transform(z_eval[eval_mask])
    y_train_scaled = y_scaler.fit_transform(y_train[train_mask])

    model = MLPRegressor(
        hidden_layer_sizes=(256, 128),
        activation="relu",
        alpha=1e-4,
        batch_size=512,
        learning_rate_init=1e-3,
        max_iter=200,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=12,
        random_state=0,
        verbose=False,
    )
    model.fit(x_train, y_train_scaled)
    pred_scaled = model.predict(x_eval).reshape(len(x_eval), -1)
    pred = y_scaler.inverse_transform(pred_scaled)
    grouped_r2, component_r2 = score_predictions(y_eval[eval_mask], pred)
    return {
        "r2": grouped_r2,
        "component_r2": component_r2,
        "n_train": int(train_mask.sum()),
        "n_eval": int(eval_mask.sum()),
    }


PROBE_FNS = {
    "linear": fit_linear_probe,
    "mlp": fit_mlp_probe,
}


In [ ]:
def compute_probe_scores():
    rows = []
    component_rows = []

    for env in ENVS:
        env_slug = env["slug"]
        for method in METHODS:
            key = (env_slug, method["name"])
            if key not in exports:
                continue
            export = exports[key]
            targets = target_inventory(env_slug, export)
            for spec, train_target, eval_target in targets:
                for probe_type in PROBE_TYPES:
                    if probe_type == "mlp" and not RUN_MLP_PROBE:
                        continue
                    print(f"[{probe_type}] {env_slug}/{method['name']}: {spec['quantity']}")
                    result = PROBE_FNS[probe_type](
                        export["train"]["z"], train_target["y"],
                        export["eval"]["z"], eval_target["y"],
                    )
                    row = {
                        "environment": env["name"],
                        "env_slug": env_slug,
                        "method": method["name"],
                        "method_label": method["label"],
                        "probe_type": probe_type,
                        "quantity": spec["quantity"],
                        "source_key": train_target["source"],
                        "secondary": bool(spec.get("secondary", False)),
                        "r2": result["r2"],
                        "component_r2_json": json.dumps(result["component_r2"]),
                        "n_train": result["n_train"],
                        "n_eval": result["n_eval"],
                    }
                    rows.append(row)

                    for component, component_r2 in zip(train_target["components"], result["component_r2"]):
                        component_rows.append({
                            **{k: row[k] for k in [
                                "environment", "env_slug", "method", "method_label",
                                "probe_type", "quantity", "source_key", "secondary",
                                "n_train", "n_eval",
                            ]},
                            "component": component,
                            "component_r2": component_r2,
                        })

    return pd.DataFrame(rows), pd.DataFrame(component_rows)


probe_scores, component_scores = compute_probe_scores()

if probe_scores is not None:
    summary_cols = [
        "environment", "method_label", "probe_type", "quantity", "source_key",
        "secondary", "r2", "n_train", "n_eval",
    ]
    display(
        probe_scores[summary_cols]
        .sort_values(["environment", "quantity", "probe_type", "method_label"])
        .reset_index(drop=True)
    )


In [ ]:
def lookup_r2(scores, env_slug, quantity, probe_type, method):
    match = scores[
        (scores["env_slug"] == env_slug)
        & (scores["quantity"] == quantity)
        & (scores["probe_type"] == probe_type)
        & (scores["method"] == method)
    ]
    if match.empty:
        print(f"[warn] missing score: {env_slug} / {quantity} / {probe_type} / {method}")
        return np.nan
    return float(match.iloc[0]["r2"])


def build_final_table(scores):
    rows = []
    for env in ENVS:
        env_slug = env["slug"]
        for quantity in DISPLAY_QUANTITY_GROUPS.get(env_slug, []):
            row = {"Environment": env["name"], "Quantity": quantity}
            for probe_type in PROBE_TYPES:
                for method in METHODS:
                    col = f"{PROBE_LABELS[probe_type]}: {method['label']}"
                    row[col] = lookup_r2(scores, env_slug, quantity, probe_type, method["name"])
            rows.append(row)
    return pd.DataFrame(rows)


def format_score(value):
    if pd.isna(value):
        return "--"
    return f"{float(value):.3f}"


def score_column_groups():
    groups = {}
    for probe_type in PROBE_TYPES:
        groups[probe_type] = [
            f"{PROBE_LABELS[probe_type]}: {method['label']}"
            for method in METHODS
        ]
    return groups


def best_score_mask(table, atol=1e-12):
    mask = pd.DataFrame(False, index=table.index, columns=table.columns)
    for cols in score_column_groups().values():
        available_cols = [col for col in cols if col in table]
        if not available_cols:
            continue
        values = table[available_cols].astype(float)
        row_max = values.max(axis=1, skipna=True)
        for col in available_cols:
            mask[col] = values[col].notna() & np.isclose(values[col], row_max, atol=atol, rtol=0.0)
    return mask


def style_final_table(display_table, numeric_table):
    bold_mask = best_score_mask(numeric_table)
    env_breaks = numeric_table["Environment"].ne(numeric_table["Environment"].shift())
    env_breaks.iloc[0] = False

    styles = pd.DataFrame("", index=display_table.index, columns=display_table.columns)
    styles = styles.mask(bold_mask, styles + "font-weight: bold;")
    for idx, is_break in env_breaks.items():
        if is_break:
            styles.loc[idx, :] = styles.loc[idx, :] + "border-top: 2px solid #b8b8b8;"
    return styles


final_table = build_final_table(probe_scores)

score_cols = [c for c in final_table.columns if c not in {"Environment", "Quantity"}]
final_display = final_table.copy()
for col in score_cols:
    final_display[col] = final_display[col].map(format_score)
final_display.loc[final_display["Environment"].duplicated(), "Environment"] = ""

display(
    final_display.style
    .hide(axis="index")
    .apply(lambda _: style_final_table(final_display, final_table), axis=None)
    .set_caption("Eval R2 for physical-quantity probes. Probes are trained on train embeddings only.")
)


In [ ]:
def latex_escape(text):
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return "".join(replacements.get(ch, ch) for ch in str(text))


def format_score_latex(row, col, bold_mask):
    value = format_score(row[col])
    if value != "--" and bool(bold_mask.loc[row.name, col]):
        return rf"\textbf{{{value}}}"
    return value


def make_latex_table(table):
    bold_mask = best_score_mask(table)
    lines = [
        r"% Requires \usepackage{booktabs} and \usepackage[table]{xcolor}",
        r"\begin{table}[t]",
        r"\centering",
        r"\caption{Eval $R^2$ for physical-quantity probes trained on frozen embeddings.}",
        r"\label{tab:physical_quantity_probe_r2}",
        r"\small",
        r"\setlength{\tabcolsep}{3.5pt}",
        r"\begin{tabular}{llrrrrrr}",
        r"\toprule",
        r"Environment & Quantity & \multicolumn{3}{c}{Linear probe} & \multicolumn{3}{c}{MLP probe} \\",
        r"\cmidrule(lr){3-5}\cmidrule(lr){6-8}",
        r" & & ours & SIGReg & forward-only & ours & SIGReg & forward-only \\",
        r"\midrule",
    ]

    previous_env = None
    for idx, row in table.iterrows():
        env = row["Environment"]
        if previous_env is not None and env != previous_env:
            lines.append(r"\arrayrulecolor{black!35}\midrule\arrayrulecolor{black}")
        env_cell = latex_escape(env) if env != previous_env else ""
        quantity = latex_escape(row["Quantity"])
        value_cols = [
            "Linear probe: ours",
            "Linear probe: SIGReg",
            "Linear probe: forward-only",
            "MLP probe: ours",
            "MLP probe: SIGReg",
            "MLP probe: forward-only",
        ]
        values = [format_score_latex(row, col, bold_mask) for col in value_cols]
        lines.append(f"{env_cell} & {quantity} & " + " & ".join(values) + r" \\")
        previous_env = env

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
    ])
    return "\n".join(lines)


latex_code = make_latex_table(final_table)
display(Markdown("```latex\n" + latex_code + "\n```"))


## Notes

- Linear and MLP probes train only on train embeddings and report only eval R2.
- MLP early stopping uses scikit-learn's internal validation split from the train data; eval data is not used for tuning.
- Angular quantities are represented with sine/cosine targets before probing.
- Velocity-like targets are computed as secondary targets where available, but are excluded from the default displayed table.
- Edit `DISPLAY_QUANTITY_GROUPS` in the config cell to change which grouped quantities appear in the final table.
